In [ ]:
# =========================
# 0A) Setup deps + ABI fixes (may require runtime restart)
# =========================
import os, sys, subprocess

def _pip(*args):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

need_restart = False

# ---- Fix NumPy ABI mismatch (NumPy 2.x often breaks compiled wheels) ----
try:
    import numpy as np
    major = int(np.__version__.split(".")[0])
except Exception:
    major = 999

if major >= 2:
    _pip("install", "-q", "--upgrade", "--force-reinstall", "numpy==1.26.4")
    need_restart = True

# ---- Core deps ----
_pip("install", "-q", "--upgrade", "gymnasium==0.29.1", "stable-baselines3==2.3.2", "shimmy>=1.3.0")

# ---- Remove gym to silence warning (only matters if gym exists) ----
try:
    import gym  # noqa
    _pip("uninstall", "-y", "gym")
    need_restart = True
except Exception:
    pass

# ---- Optional: remove torchao to avoid transformers pulling it ----
try:
    _pip("uninstall", "-y", "torchao")
except Exception:
    pass

if need_restart:
    raise SystemExit("Deps changed. Runtime -> Restart runtime now, then run Cell 0B.")
else:
    print("Deps OK. No restart needed. Next: run Cell 0B.")


+ /usr/bin/python3 -m pip install -q --upgrade --force-reinstall numpy==1.26.4
+ /usr/bin/python3 -m pip install -q --upgrade gymnasium==0.29.1 stable-baselines3==2.3.2 shimmy>=1.3.0
+ /usr/bin/python3 -m pip uninstall -y torchao


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


SystemExit: Deps changed. Runtime -> Restart runtime now, then run Cell 0B.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# =========================
# 0B) Paths + Drive mount + globals (RUNS_ROOT / DATA_DIR)  [FIXED]
# - Works with "Shared with me" if you added a shortcut into MyDrive
# - Also tries Drive's .shortcut-targets-by-id
# - Auto-finds a folder that contains weather_cache
# =========================
import os
from pathlib import Path

# ---- Detect Colab and mount Drive ----
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive"):
        drive.mount("/content/drive", force_remount=False)

# ---- Define RUNS_ROOT ----
BASE_DIR = Path("/content/drive/MyDrive/AURORA/AURORA_runs/results") if IN_COLAB else Path("./benchmarks")
RUNS_ROOT = str((BASE_DIR / "aurora_runs").resolve())
Path(RUNS_ROOT).mkdir(parents=True, exist_ok=True)

# ---- Helper: find dataset root ----
def _find_data_dir() -> str | None:
    if not IN_COLAB:
        # local fallback
        local = Path("./aurora_data")
        if local.exists():
            wc = local / "weather_cache"
            if wc.is_dir():
                return str(local.resolve())
            return str(local.resolve())
        return None

    mydrive = Path("/content/drive/MyDrive")

    # 1) If you made a shortcut into MyDrive, it will show up like a normal folder.
    # Try common names first (fast).
    candidates = [
        mydrive / "ISEF DATA",
        mydrive / "ISEF_DATA",
        mydrive / "isef data",
        mydrive / "aurora_data",
        mydrive / "Aurora_Data",
    ]
    for c in candidates:
        if c.is_dir():
            # If it has weather_cache, perfect
            if (c / "weather_cache").is_dir():
                return str(c.resolve())
            # Otherwise still accept it as a root
            return str(c.resolve())

    # 2) Search inside Drive shortcut targets (covers Shared-with-me shortcuts)
    shortcut_root = mydrive / ".shortcut-targets-by-id"
    if shortcut_root.is_dir():
        try:
            # Look for any folder containing weather_cache (best signal)
            for p in shortcut_root.rglob("weather_cache"):
                if p.is_dir():
                    return str(p.parent.resolve())
        except Exception:
            pass

        try:
            # Or look for folder name containing "ISEF"
            for p in shortcut_root.rglob("*"):
                if p.is_dir() and "ISEF" in p.name.upper():
                    if (p / "weather_cache").is_dir():
                        return str(p.resolve())
        except Exception:
            pass

    # 3) Last resort: search MyDrive shallowly for weather_cache without crawling forever
    # Only scan a few levels to stay fast.
    try:
        for p in mydrive.glob("*"):
            if p.is_dir() and (p / "weather_cache").is_dir():
                return str(p.resolve())
    except Exception:
        pass

    return None

DATA_DIR = _find_data_dir()

globals()["RUNS_ROOT"] = RUNS_ROOT
globals()["DATA_DIR"] = DATA_DIR

# Also set env var used by the runtime package
if DATA_DIR is not None:
    os.environ["AURORA_DATA_DIR"] = DATA_DIR

print("IN_COLAB:", IN_COLAB)
print("RUNS_ROOT:", RUNS_ROOT)
print("DATA_DIR:", DATA_DIR)
if DATA_DIR is None:
    print("[WARN] Could not auto-find dataset. Add a shortcut of 'ISEF DATA' into MyDrive, or set DATA_ROOT_FOR_TRAIN manually.")
else:
    print("Found weather_cache:", os.path.isdir(os.path.join(DATA_DIR, "weather_cache")))


IN_COLAB: True
RUNS_ROOT: /content/drive/MyDrive/AURORA/AURORA_runs/results/aurora_runs
DATA_DIR: /content/drive/.shortcut-targets-by-id/1C8hxZfw_g_lXxdROFuGwTbiuMcl-qAgn/ISEF DATA
Found weather_cache: True


In [ ]:
# =========================
# 2) Load Qwen (global, single-load) + qwen_generate + set_qwen_model
# - bf16 on A100/H100 when available
# - no bitsandbytes
# =========================
import os
from typing import Dict, Tuple, Optional

# Reduce optional backend imports inside transformers
os.environ["TRANSFORMERS_NO_TORCHAO"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_TOKEN = os.environ.get("HF_TOKEN", None)

MODEL_ID_3B = globals().get("MODEL_ID_3B", "Qwen/Qwen2.5-3B-Instruct")
MODEL_ID_7B = globals().get("MODEL_ID_7B", "Qwen/Qwen2.5-7B-Instruct")

# Load BOTH so the env can switch sizes without reloading.
# If your GPU is small, this will auto-disable.
LOAD_BOTH = bool(globals().get("LOAD_BOTH", True))

def _vram_gb() -> Optional[float]:
    if not torch.cuda.is_available():
        return None
    try:
        return torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception:
        return None

def _pick_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    try:
        major, _ = torch.cuda.get_device_capability(0)
        if major >= 8 and torch.cuda.is_bf16_supported():
            return torch.bfloat16
    except Exception:
        pass
    return torch.float16

def _load_one(model_id: str):
    print("\n[QWEN] loading:", model_id)
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, token=HF_TOKEN)
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token

    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=_pick_dtype(),
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    mdl.eval()
    return tok, mdl

vr = _vram_gb()
if vr is not None and vr < 24 and LOAD_BOTH:
    print(f"[QWEN] VRAM ~{vr:.1f} GB, forcing LOAD_BOTH=False to avoid OOM")
    LOAD_BOTH = False

QWEN_MODELS: Dict[str, Tuple[AutoTokenizer, AutoModelForCausalLM, str]] = {}

tok3, mdl3 = _load_one(MODEL_ID_3B)
QWEN_MODELS["3B"] = (tok3, mdl3, MODEL_ID_3B)

if LOAD_BOTH:
    tok7, mdl7 = _load_one(MODEL_ID_7B)
    QWEN_MODELS["7B"] = (tok7, mdl7, MODEL_ID_7B)

ACTIVE_QWEN_SIZE = "3B"
tokenizer, model, MODEL_ID = QWEN_MODELS[ACTIVE_QWEN_SIZE]

def set_qwen_model(size: str):
    global tokenizer, model, MODEL_ID, ACTIVE_QWEN_SIZE
    s = str(size).upper().replace(" ", "")
    if s in {"3", "3B"}:
        s = "3B"
    elif s in {"7", "7B"}:
        s = "7B"
    else:
        raise ValueError("size must be '3B' or '7B'")
    if s not in QWEN_MODELS:
        raise RuntimeError(f"Model {s} not loaded. Set LOAD_BOTH=True and have enough VRAM.")
    tokenizer, model, MODEL_ID = QWEN_MODELS[s]
    ACTIVE_QWEN_SIZE = s
    print(f"[QWEN] active -> {ACTIVE_QWEN_SIZE}: {MODEL_ID}")

@torch.no_grad()
def qwen_generate(prompt: str, max_new_tokens: int = 220) -> str:
    messages = [{"role": "user", "content": prompt}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = prompt

    inputs = tokenizer(text, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=int(max_new_tokens),
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[-1]
    gen_ids = out[0][prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

print("[QWEN] loaded:", list(QWEN_MODELS.keys()))
print("[QWEN] default active:", ACTIVE_QWEN_SIZE, MODEL_ID)



[QWEN] loading: Qwen/Qwen2.5-3B-Instruct


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[QWEN] loaded: ['3B', '7B']
[QWEN] default active: 3B Qwen/Qwen2.5-3B-Instruct


In [ ]:
# =========================
# 3) Build clean runtime package (env + agent)  [FULL FIXED CELL + PATH-CORRECT]
# - Uses data_root if passed
# - Otherwise uses os.environ["AURORA_DATA_DIR"]
# - FIX: LLM prompt forces strict JSON
# - FIX: JSON extraction uses brace-matching (much more reliable)
# - FIX: parsing falls back to ast.literal_eval for single quotes
# - FIX: prompt now includes Top-K hottest cells and constrains priority to those candidates
# - FIX: logs llm_chars / llm_words (instead of fake "token" counting)
# - BONUS: puts llm_raw_preview into info when parsing fails
# =========================
import os, sys, textwrap, shutil
from pathlib import Path

PKG_ROOT = Path("/content/aurora_clean_pkg").resolve()
PKG_NAME = "aurora_clean"

# Start clean to avoid stale files
if PKG_ROOT.exists():
    shutil.rmtree(PKG_ROOT, ignore_errors=True)

def _write(rel_path: str, content: str):
    p = PKG_ROOT / rel_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(content), encoding="utf-8")
    return p

# Package init files
_write(f"{PKG_NAME}/__init__.py", "from .env.training_env import HybridRealFireEnv\n")
_write(f"{PKG_NAME}/env/__init__.py", "")
_write(f"{PKG_NAME}/agents/__init__.py", "")
_write(f"{PKG_NAME}/data/__init__.py", "")

# -------------------------
# FireSim: simple spread model
# -------------------------
_write(f"{PKG_NAME}/env/fire_sim.py", r'''
import numpy as np

class FireSim:
    def __init__(self, grid_size: int = 50):
        self.grid_size = int(grid_size)
        self.fire_grid = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)

    def reset(self, initial_grid: np.ndarray):
        self.fire_grid = initial_grid.astype(np.float32).copy()

    def step(self, wind=(0.0, 0.0), suppression=None):
        g = self.fire_grid

        neigh = (
            np.roll(g, 1, 0) + np.roll(g, -1, 0) +
            np.roll(g, 1, 1) + np.roll(g, -1, 1)
        ) / 4.0

        wx, wy = float(wind[0]), float(wind[1])

        if wx > 0:
            neigh = 0.7 * neigh + 0.3 * np.roll(neigh, 1, 1)
        elif wx < 0:
            neigh = 0.7 * neigh + 0.3 * np.roll(neigh, -1, 1)

        if wy > 0:
            neigh = 0.7 * neigh + 0.3 * np.roll(neigh, 1, 0)
        elif wy < 0:
            neigh = 0.7 * neigh + 0.3 * np.roll(neigh, -1, 0)

        ignite = (neigh > 0.08).astype(np.float32) * 0.06
        burn = g * 0.01

        g2 = np.clip(g + ignite - burn, 0.0, 1.0)

        if suppression is not None:
            g2 = np.clip(g2 - suppression.astype(np.float32) * 0.35, 0.0, 1.0)

        self.fire_grid = g2
''')

# -------------------------
# Drone agent: simple kinematics
# -------------------------
_write(f"{PKG_NAME}/agents/drone_agent.py", r'''
class DroneAgent:
    # actions: 0 suppress, 1 up, 2 down, 3 left, 4 right, 5 noop
    def __init__(self, x: int, y: int, grid_size: int = 50):
        self.grid_size = int(grid_size)
        self.x = int(x)
        self.y = int(y)

    def step(self, action: int):
        a = int(action)
        if a == 1:
            self.y = max(0, self.y - 1)
        elif a == 2:
            self.y = min(self.grid_size - 1, self.y + 1)
        elif a == 3:
            self.x = max(0, self.x - 1)
        elif a == 4:
            self.x = min(self.grid_size - 1, self.x + 1)
        # 0 and 5 do not change position
''')

# -------------------------
# RealDataIntegrator: safe loader (drive/dataset friendly)
# - If data_root is None, uses env var AURORA_DATA_DIR
# -------------------------
_write(f"{PKG_NAME}/data/real_data_integration_complete.py", r'''
import os
import json
import numpy as np

class RealDataIntegrator:
    """
    Safe, Drive-friendly data integrator.

    Reads wind from:
      - {data_root}/weather_cache/**/*.json (preferred)
      - or falls back to synthetic weather.

    IMPORTANT:
      - If data_root is None, uses env var AURORA_DATA_DIR.
    """
    _printed_root = False

    def __init__(self, data_root: str = None, grid_size: int = 50):
        env_root = os.environ.get("AURORA_DATA_DIR") or os.environ.get("DATA_ROOT")
        self.data_root = data_root if data_root not in (None, "", "none", "null") else env_root
        self.grid_size = int(grid_size)

        if not RealDataIntegrator._printed_root:
            RealDataIntegrator._printed_root = True
            print("[RealDataIntegrator] data_root =", self.data_root)

        self._weather_files = []
        if self.data_root:
            wc = os.path.join(self.data_root, "weather_cache")
            if os.path.isdir(wc):
                try:
                    for root, _, files in os.walk(wc):
                        for fn in files:
                            if fn.lower().endswith(".json"):
                                self._weather_files.append(os.path.join(root, fn))
                except Exception:
                    self._weather_files = []

    def _extract_wind(self, obj: dict) -> dict:
        ws = None
        for k in ("wind_speed_mph", "wind_mph", "wind_speed", "windSpeedMph", "windspeed_mph"):
            if k in obj:
                ws = obj.get(k)
                break

        wd = None
        for k in ("wind_direction", "wind_dir", "wind_direction_deg", "windDir", "windDirection"):
            if k in obj:
                wd = obj.get(k)
                break

        out = {}
        if ws is not None:
            try:
                out["wind_speed_mph"] = float(ws)
            except Exception:
                pass

        if wd is not None:
            if isinstance(wd, dict):
                wd = wd.get("deg", wd.get("degrees", wd.get("value", wd)))
            out["wind_direction"] = wd

        return out

    def _sample_weather(self, rng: np.random.Generator) -> dict:
        if self._weather_files:
            try:
                path = self._weather_files[int(rng.integers(0, len(self._weather_files)))]
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    obj = json.load(f)

                if isinstance(obj, dict) and "data" in obj and isinstance(obj["data"], dict):
                    obj_use = obj["data"]
                else:
                    obj_use = obj

                out = self._extract_wind(obj_use)
                if "wind_speed_mph" not in out:
                    out["wind_speed_mph"] = float(rng.uniform(2.0, 18.0))
                if "wind_direction" not in out:
                    out["wind_direction"] = str(rng.choice(["N","NE","E","SE","S","SW","W","NW"]))
                return out
            except Exception:
                pass

        return {
            "wind_speed_mph": float(rng.uniform(2.0, 18.0)),
            "wind_direction": str(rng.choice(["N","NE","E","SE","S","SW","W","NW"]))
        }

    def create_training_scenario(self, rng: np.random.Generator):
        g = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)

        k = int(rng.integers(3, 8))
        for _ in range(k):
            cy = int(rng.integers(0, self.grid_size))
            cx = int(rng.integers(0, self.grid_size))
            rad = int(rng.integers(2, 6))
            y0, y1 = max(0, cy - rad), min(self.grid_size, cy + rad + 1)
            x0, x1 = max(0, cx - rad), min(self.grid_size, cx + rad + 1)
            g[y0:y1, x0:x1] = np.maximum(g[y0:y1, x0:x1], rng.uniform(0.4, 1.0))

        for _ in range(2):
            g = (
                g
                + np.roll(g, 1, 0) + np.roll(g, -1, 0)
                + np.roll(g, 1, 1) + np.roll(g, -1, 1)
            ) / 5.0
            g = np.clip(g, 0.0, 1.0)

        return {
            "initial_fire_grid": g,
            "weather": self._sample_weather(rng),
        }
''')

# -------------------------
# HybridPPOLLMAgent: shared Qwen bridge (no model loading here)
# - strict JSON output
# - brace-matching extraction (reliable)
# - fallback parsing for single quotes
# - includes Top-K hottest cells and constrains priority to candidates
# - logs llm_chars / llm_words
# -------------------------
_write(f"{PKG_NAME}/agents/hybrid_ppo_llm_agent.py", r'''
import time, json, ast
from typing import Any, Dict, List, Tuple
import numpy as np

class HybridPPOLLMAgent:
    def __init__(self, llm_model: str = "none", llm_guidance_frequency: int = 0, topk: int = 12):
        self.llm_model = str(llm_model)
        self.llm_guidance_frequency = int(llm_guidance_frequency)
        self.topk = int(topk)
        self._last = None
        self._calls = 0

    @property
    def calls(self) -> int:
        return int(self._calls)

    def _strip_fences(self, s: str) -> str:
        s = (s or "").strip()
        s = s.replace("```json", "```").replace("```JSON", "```")
        if "```" in s:
            s = s.replace("```", "").strip()
        return s

    def _first_complete_object(self, text: str) -> str:
        s = self._strip_fences(str(text))

        i = s.find("{")
        if i == -1:
            return ""

        depth = 0
        in_str = False
        esc = False

        for j in range(i, len(s)):
            ch = s[j]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
                continue
            else:
                if ch == '"':
                    in_str = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        return s[i:j+1]

        return ""

    def _topk_hot_cells(self, fire_state: np.ndarray, k: int):
        g = np.asarray(fire_state, dtype=np.float32)
        H, W = g.shape
        flat = g.reshape(-1)
        k = int(max(1, min(k, flat.size)))

        idx = np.argpartition(flat, -k)[-k:]
        idx = idx[np.argsort(flat[idx])[::-1]]

        out = []
        for ii in idx:
            r = int(ii // W)
            c = int(ii % W)
            out.append([r, c, float(flat[ii])])
        return out

    def _pick_fallback(self, top_cells: List[List[float]], drone_positions: List[Tuple[int,int]]):
        # drone_positions are (x,y); candidates are [row,col] == [y,x]
        occ = set((int(x), int(y)) for (x, y) in drone_positions)
        for r, c, _v in top_cells:
            if (int(c), int(r)) not in occ:
                return [int(r), int(c)]
        # if all occupied, just pick hottest
        return [int(top_cells[0][0]), int(top_cells[0][1])]

    def maybe_get_guidance(
        self,
        fire_state: np.ndarray,
        drone_positions: List[Tuple[int, int]],
        weather: Dict[str, Any],
        step: int,
    ):
        info = {"llm_called": False, "llm_latency_ms": None, "llm_chars": None, "llm_words": None}

        if self.llm_model.lower() in {"none", "null", ""} or self.llm_guidance_frequency <= 0:
            return None, info

        if step % self.llm_guidance_frequency != 0:
            return self._last, info

        try:
            import __main__
            qgen = getattr(__main__, "qwen_generate", None)
            qset = getattr(__main__, "set_qwen_model", None)
        except Exception:
            qgen, qset = None, None

        if qgen is None:
            return self._last, info

        if qset is not None:
            try:
                qset("7B" if "7b" in self.llm_model.lower() else "3B")
            except Exception:
                pass

        fire_state = np.asarray(fire_state, dtype=np.float32)
        H, W = int(fire_state.shape[0]), int(fire_state.shape[1])
        density = float(fire_state.mean())
        top_cells = self._topk_hot_cells(fire_state, k=self.topk)

        # STRICT JSON prompt with explicit candidate constraint
        prompt = (
            "Return ONLY valid JSON. No prose. No markdown. No code fences.\n"
            "Schema exactly:\n"
            "{\"priority\":[row,col],\"hint\":\"short string\"}\n"
            "Hard rules:\n"
            "- priority MUST be one of the [row,col] pairs listed in TopCells\n"
            "- prefer the highest-intensity TopCells entry not occupied by a drone\n"
            "- hint must be <= 12 words and reference the chosen cell\n"
            f"Step: {int(step)}\n"
            f"GridSize: {H}x{W}\n"
            f"FireDensity: {density:.3f}\n"
            f"Drones(x,y): {drone_positions}\n"
            f"TopCells([row,col,intensity]): {top_cells}\n"
            f"WindSpeedMph: {weather.get('wind_speed_mph', 0):.1f}\n"
            f"WindDir: {weather.get('wind_direction', 'N')}\n"
        )

        t0 = time.time()
        try:
            out = qgen(prompt, max_new_tokens=120, temperature=0.0, do_sample=False, top_p=1.0)
        except TypeError:
            out = qgen(prompt, max_new_tokens=120)
        dt_ms = (time.time() - t0) * 1000.0

        out_str = str(out)
        info["llm_called"] = True
        info["llm_latency_ms"] = float(dt_ms)
        info["llm_chars"] = int(len(out_str))
        info["llm_words"] = int(len(out_str.split()))

        obj = self._first_complete_object(out_str)
        parsed = None

        if obj:
            try:
                parsed = json.loads(obj)
            except Exception:
                parsed = None
            if parsed is None:
                try:
                    parsed = ast.literal_eval(obj)
                except Exception:
                    parsed = None

        if not isinstance(parsed, dict):
            info["llm_raw_preview"] = out_str[:400]
            return self._last, info

        if "priority" not in parsed or "hint" not in parsed:
            info["llm_raw_preview"] = out_str[:400]
            return self._last, info

        # Normalize priority
        pr = parsed.get("priority")
        rr, cc = None, None
        if isinstance(pr, (list, tuple)) and len(pr) == 2:
            try:
                rr, cc = int(pr[0]), int(pr[1])
            except Exception:
                rr, cc = None, None

        # Build candidate set and validate constraint: must be in TopCells
        cand = set((int(r), int(c)) for r, c, _v in top_cells)
        if rr is None or cc is None or (rr, cc) not in cand:
            rr, cc = self._pick_fallback(top_cells, drone_positions)

        # Bounds safety (should already hold if in cand)
        rr = int(max(0, min(H - 1, rr)))
        cc = int(max(0, min(W - 1, cc)))

        parsed["priority"] = [rr, cc]
        parsed["hint"] = str(parsed.get("hint", ""))[:200]

        self._calls += 1
        self._last = parsed
        return self._last, info
''')

# -------------------------
# Training env: gymnasium env
# -------------------------
_write(f"{PKG_NAME}/env/training_env.py", r'''
import math
import numpy as np

try:
    import gymnasium as gym
    from gymnasium import spaces
except Exception:
    import gym
    from gym import spaces

from .fire_sim import FireSim
from ..agents.drone_agent import DroneAgent
from ..agents.hybrid_ppo_llm_agent import HybridPPOLLMAgent
from ..data.real_data_integration_complete import RealDataIntegrator

_COMPASS = {
    "N":  (0.0, -1.0),
    "NE": (math.sqrt(0.5), -math.sqrt(0.5)),
    "E":  (1.0, 0.0),
    "SE": (math.sqrt(0.5), math.sqrt(0.5)),
    "S":  (0.0, 1.0),
    "SW": (-math.sqrt(0.5), math.sqrt(0.5)),
    "W":  (-1.0, 0.0),
    "NW": (-math.sqrt(0.5), -math.sqrt(0.5)),
}

def _dir_to_unit(d):
    if isinstance(d, (int, float)):
        deg = float(d) % 360.0
        rad = math.radians(deg)
        return (math.cos(rad), math.sin(rad))
    s = str(d).upper().strip()
    if s in _COMPASS:
        return _COMPASS[s]
    try:
        deg = float(s) % 360.0
        rad = math.radians(deg)
        return (math.cos(rad), math.sin(rad))
    except Exception:
        return _COMPASS["N"]

def _wind_vec(weather: dict):
    sp = float(weather.get("wind_speed_mph", 0.0))
    d = weather.get("wind_direction", "N")
    ux, uy = _dir_to_unit(d)
    scale = sp / 20.0
    return (ux * scale, uy * scale)

def _downsample(grid: np.ndarray, k: int):
    H, W = grid.shape
    hh = H // k
    ww = W // k
    g = grid[:hh*k, :ww*k]
    g = g.reshape(hh, k, ww, k).mean(axis=(1,3))
    return g.astype(np.float32)

class HybridRealFireEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        grid_size: int = 50,
        num_drones: int = 3,
        max_steps: int = 200,
        llm_model: str = "none",
        llm_guidance_freq: int = 0,
        data_root: str = None,
        downsample_k: int = 5,
    ):
        super().__init__()
        self.grid_size = int(grid_size)
        self.num_drones = int(num_drones)
        self.max_steps = int(max_steps)
        self.downsample_k = int(downsample_k)

        self.rng = np.random.default_rng(0)
        self.integrator = RealDataIntegrator(data_root=data_root, grid_size=self.grid_size)
        self.fire_sim = FireSim(self.grid_size)
        self.hybrid_agent = HybridPPOLLMAgent(llm_model=llm_model, llm_guidance_frequency=llm_guidance_freq)

        hh = self.grid_size // self.downsample_k
        ww = self.grid_size // self.downsample_k
        fire_dim = int(hh * ww)

        obs_dim = fire_dim + (2 * self.num_drones) + 1 + 2
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([6] * self.num_drones)

        self._step = 0
        self._scenario = None
        self._drones = None
        self._last_info = {}

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self._scenario = self.integrator.create_training_scenario(self.rng)
        self.fire_sim.reset(self._scenario["initial_fire_grid"])

        spawns = [(0,0), (self.grid_size-1,0), (0,self.grid_size-1), (self.grid_size-1,self.grid_size-1)]
        self._drones = [DroneAgent(*spawns[i % len(spawns)], grid_size=self.grid_size) for i in range(self.num_drones)]
        self._step = 0
        self._last_info = {}
        return self._obs(), {}

    def _obs(self):
        fire_ds = _downsample(self.fire_sim.fire_grid, k=self.downsample_k).reshape(-1)

        drone_xy = []
        for d in self._drones:
            drone_xy.extend([d.x / (self.grid_size - 1), d.y / (self.grid_size - 1)])
        drone_xy = np.array(drone_xy, dtype=np.float32)

        prog = np.array([self._step / max(self.max_steps, 1)], dtype=np.float32)

        w = _wind_vec(self._scenario["weather"])
        wind = np.array(
            [
                np.clip((w[0] + 1.0) / 2.0, 0.0, 1.0),
                np.clip((w[1] + 1.0) / 2.0, 0.0, 1.0),
            ],
            dtype=np.float32,
        )

        return np.concatenate([fire_ds, drone_xy, prog, wind], axis=0).astype(np.float32)

    def step(self, actions):
        self._step += 1

        sup = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        for d, a in zip(self._drones, actions):
            a = int(a)
            d.step(a)
            if a == 0:
                y, x = d.y, d.x
                sup[max(0,y-1):min(self.grid_size,y+2), max(0,x-1):min(self.grid_size,x+2)] = 1.0

        guidance, llm_info = self.hybrid_agent.maybe_get_guidance(
            self.fire_sim.fire_grid,
            [(d.x, d.y) for d in self._drones],
            self._scenario["weather"],
            self._step,
        )

        wind = _wind_vec(self._scenario["weather"])
        self.fire_sim.step(wind=wind, suppression=sup)

        fire_cov = float(self.fire_sim.fire_grid.mean())
        containment = 1.0 - fire_cov
        reward = float((containment * 2.0) - (0.002 * self._step))

        terminated = bool(fire_cov < 0.005)
        truncated = bool(self._step >= self.max_steps)

        info = {
            "containment": float(containment),
            "fire_coverage": float(fire_cov),
            "fire_size": float(self.fire_sim.fire_grid.sum()),
        }
        info.update(llm_info)
        info["llm_guidance_present"] = bool(isinstance(guidance, dict))

        self._last_info = info
        return self._obs(), reward, terminated, truncated, info
''')

# Put pkg on path
PKG_ROOT.mkdir(parents=True, exist_ok=True)
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

# Purge cached modules so the very next import uses the rewritten files
for m in list(sys.modules.keys()):
    if m.startswith("aurora_clean"):
        del sys.modules[m]

print("Wrote clean runtime package to:", PKG_ROOT)
print("sys.path[0]:", sys.path[0])
print("AURORA_DATA_DIR (env):", os.environ.get("AURORA_DATA_DIR"))
print("Purged aurora_clean modules. Next import will use fresh code.")


Wrote clean runtime package to: /content/aurora_clean_pkg
sys.path[0]: /content/aurora_clean_pkg
AURORA_DATA_DIR (env): /content/drive/.shortcut-targets-by-id/1C8hxZfw_g_lXxdROFuGwTbiuMcl-qAgn/ISEF DATA
Purged aurora_clean modules. Next import will use fresh code.


In [ ]:
# =========================
# 4B) Smoke test WITH LLM (verifies qwen_generate is used)  [FIXED]
# - Forces a fresh import of aurora_clean so you definitely use the rewritten files from Cell 3
# - Prints LLM call + guidance parsing status
# - If parsing fails, prints llm_raw_preview (so you can see what Qwen actually returned)
# =========================
import sys, os
import numpy as np

# Force fresh imports (important after rewriting files in Cell 3)
for m in list(sys.modules.keys()):
    if m.startswith("aurora_clean"):
        del sys.modules[m]

from aurora_clean.env.training_env import HybridRealFireEnv
import aurora_clean as _ac
import __main__

print("aurora_clean loaded from:", os.path.dirname(_ac.__file__))
print("AURORA_DATA_DIR:", os.environ.get("AURORA_DATA_DIR"))
print("HAS qwen_generate:", hasattr(__main__, "qwen_generate"))
print("HAS set_qwen_model:", hasattr(__main__, "set_qwen_model"))

env = HybridRealFireEnv(
    grid_size=50,
    num_drones=3,
    max_steps=10,
    llm_model="qwen-7b",     # or "qwen-3b"
    llm_guidance_freq=1,     # call LLM every step
    data_root=None,          # uses AURORA_DATA_DIR
)

obs, info = env.reset(seed=0)
print("obs shape:", obs.shape)
print("action_space:", env.action_space)

for t in range(3):
    actions = np.array([5, 5, 5], dtype=np.int64)  # noop
    obs, rew, term, trunc, info = env.step(actions)

    print(
        f"step {t}",
        "rew=", round(float(rew), 3),
        "llm_called=", info.get("llm_called"),
        "llm_latency_ms=", info.get("llm_latency_ms"),
        "llm_tokens=", info.get("llm_tokens"),
        "guidance_present=", info.get("llm_guidance_present"),
    )

    # Print parsed guidance if present
    try:
        g = env.hybrid_agent._last
        if isinstance(g, dict):
            print("  guidance:", g)
    except Exception:
        pass

    # If parsing failed, show the first chunk of raw LLM output
    if not info.get("llm_guidance_present"):
        rp = info.get("llm_raw_preview")
        if rp:
            print("  llm_raw_preview:", rp)

env.close()


aurora_clean loaded from: /content/aurora_clean_pkg/aurora_clean
AURORA_DATA_DIR: /content/drive/.shortcut-targets-by-id/1C8hxZfw_g_lXxdROFuGwTbiuMcl-qAgn/ISEF DATA
HAS qwen_generate: True
HAS set_qwen_model: True
[RealDataIntegrator] data_root = /content/drive/.shortcut-targets-by-id/1C8hxZfw_g_lXxdROFuGwTbiuMcl-qAgn/ISEF DATA


The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


obs shape: (109,)
action_space: MultiDiscrete([6 6 6])
[QWEN] active -> 7B: Qwen/Qwen2.5-7B-Instruct
step 0 rew= 1.739 llm_called= True llm_latency_ms= 1980.9393882751465 llm_tokens= None guidance_present= True
  guidance: {'priority': [46, 33], 'hint': 'High intensity fire southwest corner'}
[QWEN] active -> 7B: Qwen/Qwen2.5-7B-Instruct
step 1 rew= 1.711 llm_called= True llm_latency_ms= 650.8152484893799 llm_tokens= None guidance_present= True
  guidance: {'priority': [46, 33], 'hint': 'High intensity fire at top right'}
[QWEN] active -> 7B: Qwen/Qwen2.5-7B-Instruct
step 2 rew= 1.681 llm_called= True llm_latency_ms= 547.3475456237793 llm_tokens= None guidance_present= True
  guidance: {'priority': [46, 33], 'hint': 'Highest intensity fire'}


In [ ]:
# =========================
# 5) Training sweep (resume-safe, FASTSKIP, checkpoint-based completion)  [FULL FIXED CELL v4.6 - ALWAYS-CREATE-PRIMARY + MIRROR]
# - Order: baseline PPO -> Qwen 3B -> Qwen 7B
# - Defaults: 4 seeds, 1,000,000 steps
# - FASTSKIP: if final_model.zip exists, skip immediately (no env build)
# - Completion: if latest checkpoint step >= TOTAL_STEPS, skip (no env build)
# - Resume: pointer JSON -> scan latest checkpoint zip
# - Does NOT use episodes.csv for resume decisions
# - Fix: disables SB3 Rich progress bar to avoid RecursionError
# - Fix: correct Drive layout and no more "results/results/..." root bugs
# - Extra fix: will search multiple possible RUNS_ROOTs and resume from wherever checkpoints actually are
# - NEW FIX (your issue): ALWAYS creates the PRIMARY seed folder under RUNS_ROOT/results/... even if resuming/skipping from another root
#   and mirrors the best checkpoint/final_model into the PRIMARY folder so you see it in Drive immediately.
# =========================

import os, time, csv, json, re, traceback, shutil, sys, platform
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import torch

# ---- HARD DISABLE RICH (prevents recursion crash from SB3 progress bar in notebooks) ----
os.environ["RICH_DISABLE"] = "1"
os.environ["SB3_NO_PROGRESS_BAR"] = "1"

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

from aurora_clean.env.training_env import HybridRealFireEnv

def _ts():
    return time.strftime("%H:%M:%S")

def DBG(msg):
    print(f"[DBG {_ts()}] {msg}", flush=True)

# ============================================================
# ROOT PICKING (FIXED)
# - Canonical layout:
#   /content/drive/MyDrive/Aurora/AURORA_runs/
#       results/<experiment>/seed_<seed>/checkpoints/...
#       tensorboard/<experiment>/seed_<seed>/...
# ============================================================

RUNS_ROOT_USER = globals().get("RUNS_ROOT", None)

CANONICAL_1 = "/content/drive/MyDrive/Aurora/AURORA_runs"
CANONICAL_2 = "/content/drive/MyDrive/AURORA/AURORA_runs"
ALT1        = "/content/drive/MyDrive/world_models/benchmarks/aurora_runs"
ALT2        = "/content/drive/MyDrive/AURORA_runs"

def _norm_root(p: Optional[str]) -> Optional[str]:
    if not isinstance(p, str) or not p.strip():
        return None
    p = p.strip().rstrip("/")

    for cut in ["/results", "/tensorboard"]:
        idx = p.lower().find(cut)
        if idx != -1:
            p = p[:idx]

    base = os.path.basename(p).lower()
    if base == "aurora":
        p = p + "/AURORA_runs"

    if os.path.basename(p).lower() == "results":
        p = os.path.dirname(p)

    return p.rstrip("/")

def _count_existing_runs(results_root: str) -> Tuple[int, int]:
    exp_count, seed_count = 0, 0
    try:
        if not os.path.isdir(results_root):
            return 0, 0
        for exp_ent in os.scandir(results_root):
            if not exp_ent.is_dir():
                continue
            exp_count += 1
            try:
                for s_ent in os.scandir(exp_ent.path):
                    if s_ent.is_dir() and os.path.basename(s_ent.path).startswith("seed_"):
                        seed_count += 1
            except Exception:
                pass
    except Exception:
        return 0, 0
    return exp_count, seed_count

def _root_score(root: str) -> int:
    score = 0
    if not root:
        return -10**9
    if os.path.isdir(root):
        score += 100
    r = os.path.join(root, "results")
    t = os.path.join(root, "tensorboard")
    if os.path.isdir(r):
        score += 200
        exp_count, seed_count = _count_existing_runs(r)
        score += 10 * exp_count + 1 * seed_count
    if os.path.isdir(t):
        score += 10
    if root.endswith("/AURORA_runs"):
        score += 5
    return score

def _pick_runs_root(user_root: Optional[str]) -> Tuple[str, List[str]]:
    cand_raw = [
        _norm_root(user_root),
        _norm_root(CANONICAL_1),
        _norm_root(CANONICAL_2),
        _norm_root(ALT2),
        _norm_root(ALT1),
    ]
    seen = set()
    cands = []
    for c in cand_raw:
        if not c:
            continue
        if c in seen:
            continue
        seen.add(c)
        cands.append(c)

    if not cands:
        return CANONICAL_1, [CANONICAL_1]

    scored = [(c, _root_score(c)) for c in cands]
    scored_sorted = sorted(scored, key=lambda x: x[1], reverse=True)

    DBG("RUNS_ROOT candidate scores:")
    for c, s in scored_sorted:
        DBG(f"  score={s:5d} root={c}")

    best = scored_sorted[0][0]
    return best, cands

RUNS_ROOT, ALL_ROOT_CANDIDATES = _pick_runs_root(RUNS_ROOT_USER)

RESULTS_ROOT = os.path.join(RUNS_ROOT, "results")
TENSORBOARD_ROOT = os.path.join(RUNS_ROOT, "tensorboard")

os.makedirs(RUNS_ROOT, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(TENSORBOARD_ROOT, exist_ok=True)

DBG(f"RUNS_ROOT picked = {RUNS_ROOT}")
DBG(f"RESULTS_ROOT    = {RESULTS_ROOT}")
DBG(f"TENSORBOARD_ROOT= {TENSORBOARD_ROOT}")

# ---- Global knobs (paper metadata) ----
ROBUSTNESS_CHECK = bool(globals().get("ROBUSTNESS_CHECK", True))
CONVERGENCE_CONTAINMENT_THRESH = float(globals().get("CONVERGENCE_CONTAINMENT_THRESH", 0.90))
TENSORBOARD_LOG = bool(globals().get("TENSORBOARD_LOG", False))
ALLOW_FRESH_IF_LOAD_FAIL = bool(globals().get("ALLOW_FRESH_IF_LOAD_FAIL", False))

ABLATIONS = globals().get("ABLATIONS", {})
if not isinstance(ABLATIONS, dict):
    ABLATIONS = {"note": str(ABLATIONS)}

RUN_NOTES = str(globals().get("RUN_NOTES", "")).strip()

# ---- Defaults ----
TOTAL_STEPS = int(globals().get("TOTAL_STEPS", 1_000_000))
SEEDS = list(globals().get("SEEDS", [1001, 2002, 3003, 4004]))
N_ENVS = int(globals().get("N_ENVS", 1))

CHECKPOINT_EVERY_STEPS = int(globals().get("CHECKPOINT_EVERY_STEPS", 10_000))
PINNED_PRINT_EVERY_STEPS = int(globals().get("PINNED_PRINT_EVERY_STEPS", 10_000))
LLM_FREQS = list(globals().get("LLM_FREQS", [50, 100, 200]))

DATA_ROOT_FOR_TRAIN = globals().get("DATA_ROOT_FOR_TRAIN", None)

# ---- Env + PPO config (captured into run_metadata.json) ----
ENV_KWARGS = dict(
    grid_size=int(globals().get("GRID_SIZE", 50)),
    num_drones=int(globals().get("NUM_DRONES", 3)),
    max_steps=int(globals().get("MAX_EPISODE_STEPS", 200)),
)

PPO_HYPERPARAMS = dict(
    n_steps=int(globals().get("PPO_N_STEPS", 2048)),
    batch_size=int(globals().get("PPO_BATCH_SIZE", 128)),
    n_epochs=int(globals().get("PPO_N_EPOCHS", 10)),
    learning_rate=float(globals().get("PPO_LR", 3e-4)),
    gamma=float(globals().get("PPO_GAMMA", 0.995)),
    gae_lambda=float(globals().get("PPO_GAE_LAMBDA", 0.95)),
    clip_range=float(globals().get("PPO_CLIP_RANGE", 0.2)),
    ent_coef=float(globals().get("PPO_ENT_COEF", 0.01)),
    vf_coef=float(globals().get("PPO_VF_COEF", 0.5)),
    max_grad_norm=float(globals().get("PPO_MAX_GRAD_NORM", 0.5)),
)

DBG(
    f"Config: TOTAL_STEPS={TOTAL_STEPS:,} SEEDS={SEEDS} N_ENVS={N_ENVS} "
    f"CHECKPOINT_EVERY_STEPS={CHECKPOINT_EVERY_STEPS:,} LLM_FREQS={LLM_FREQS} "
    f"DATA_ROOT_FOR_TRAIN={DATA_ROOT_FOR_TRAIN}"
)
DBG(f"Paper meta: ROBUSTNESS_CHECK={ROBUSTNESS_CHECK} CONV_THRESH={CONVERGENCE_CONTAINMENT_THRESH} TB={TENSORBOARD_LOG}")
DBG(f"Safety: ALLOW_FRESH_IF_LOAD_FAIL={ALLOW_FRESH_IF_LOAD_FAIL}")

# ---- Experiments (baseline -> 3B -> 7B) ----
EXPERIMENTS = [
    {"name": "ppo_baseline_no_llm", "llm_model_id": "none", "model_size": "none", "llm_freq": 0, "ckpt_prefix": "aurora_ppo_none"},
]
for freq in LLM_FREQS:
    freq = int(freq)
    EXPERIMENTS.append({
        "name": f"ppo_llm_qwen2_5_3b_freq{freq}",
        "llm_model_id": "Qwen/Qwen2.5-3B-Instruct",
        "model_size": "3B",
        "llm_freq": freq,
        "ckpt_prefix": f"aurora_ppo_qwen3b_f{freq}"
    })
    EXPERIMENTS.append({
        "name": f"ppo_llm_qwen2_5_7b_freq{freq}",
        "llm_model_id": "Qwen/Qwen2.5-7B-Instruct",
        "model_size": "7B",
        "llm_freq": freq,
        "ckpt_prefix": f"aurora_ppo_qwen7b_f{freq}"
    })

# ---- Skip LLM exps if qwen_generate is not loaded ----
import __main__
HAS_QWEN = getattr(__main__, "qwen_generate", None) is not None
if not HAS_QWEN:
    DBG("[WARN] qwen_generate not found. Skipping all LLM experiments.")
    EXPERIMENTS = [e for e in EXPERIMENTS if e["llm_model_id"] == "none"]

MIN_FINAL_ZIP_BYTES = 10_000

def file_exists_nonempty(path: str, min_bytes: int = MIN_FINAL_ZIP_BYTES) -> bool:
    try:
        return os.path.exists(path) and os.path.getsize(path) >= min_bytes
    except Exception:
        return False

def safe_json_load_file(path: str, max_bytes: int = 200_000):
    from json import JSONDecoder
    with open(path, "rb") as f:
        data = f.read(max_bytes + 1)
    if len(data) > max_bytes:
        raise ValueError("json too large")
    return JSONDecoder().decode(data.decode("utf-8", errors="ignore"))

def parse_step_from_ckpt_name(filename: str, ckpt_prefix: str):
    m = re.match(rf"^{re.escape(ckpt_prefix)}_(\d+)_steps\.zip$", filename)
    if not m:
        return None
    try:
        return int(m.group(1))
    except Exception:
        return None

def pick_latest_checkpoint(ckpt_dir: str, ckpt_prefix: str, pointer_path: str):
    """
    Returns: (best_path, best_step) or (None, -1)
    Priority: pointer -> scan for max step
    """
    if os.path.exists(pointer_path):
        try:
            rec = safe_json_load_file(pointer_path)
            p = rec.get("path")
            if p and os.path.exists(p):
                step = parse_step_from_ckpt_name(os.path.basename(p), ckpt_prefix)
                if step is None:
                    try:
                        step = int(rec.get("step", -1))
                    except Exception:
                        step = -1
                return p, step
        except Exception:
            pass

    best_path, best_step = None, -1
    try:
        if not os.path.isdir(ckpt_dir):
            return None, -1
        for ent in os.scandir(ckpt_dir):
            if not ent.is_file():
                continue
            step = parse_step_from_ckpt_name(ent.name, ckpt_prefix)
            if step is None:
                continue
            if step > best_step:
                best_step = step
                best_path = ent.path
    except FileNotFoundError:
        return None, -1

    return best_path, best_step

# -------------------------------
# NEW: Always write into PRIMARY, but resume/skip from anywhere and mirror into PRIMARY so you SEE the folders
# -------------------------------
def primary_run_dir(exp_name: str, seed: int) -> str:
    return os.path.join(RUNS_ROOT, "results", exp_name, f"seed_{seed}")

def find_best_existing_run_dir_any_root(exp_name: str, seed: int, ckpt_prefix: str) -> str:
    """
    Finds best existing run dir across ALL_ROOT_CANDIDATES.
    Prefers:
      1) final_model.zip exists
      2) else highest checkpoint step
    Returns a path under some candidate root (might be the primary).
    """
    best_dir = primary_run_dir(exp_name, seed)
    best_score = -1

    for root in ALL_ROOT_CANDIDATES:
        rd = os.path.join(root, "results", exp_name, f"seed_{seed}")
        final_zip = os.path.join(rd, "final_model.zip")
        ckpt_dir = os.path.join(rd, "checkpoints")
        pointer = os.path.join(ckpt_dir, "latest_checkpoint.json")

        if file_exists_nonempty(final_zip, MIN_FINAL_ZIP_BYTES):
            score = 10_000_000
        else:
            ckpt_path, ckpt_step = pick_latest_checkpoint(ckpt_dir, ckpt_prefix, pointer)
            score = 0 if ckpt_path is None else (1_000_000 + int(ckpt_step))

        if score > best_score:
            best_score = score
            best_dir = rd

    return best_dir

def _copy_if_better(src: str, dst: str):
    try:
        if not os.path.exists(src):
            return
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            return
        # overwrite only if src is larger (crude but safe for csv/zip/json)
        if os.path.getsize(src) > os.path.getsize(dst):
            shutil.copy2(src, dst)
    except Exception:
        pass

def mirror_best_state_to_primary(existing_dir: str, primary_dir: str, ckpt_prefix: str):
    """
    Mirrors minimal "state" from an existing run dir to the primary run dir:
      - final_model.zip (if present)
      - episodes.csv, run_metadata.json (if present)
      - best checkpoint zip + updates latest_checkpoint.json to point at the PRIMARY checkpoint path
    This is the key fix so Drive immediately shows seed folders under the primary root.
    """
    if existing_dir == primary_dir:
        return

    os.makedirs(primary_dir, exist_ok=True)
    ex_final = os.path.join(existing_dir, "final_model.zip")
    ex_csv   = os.path.join(existing_dir, "episodes.csv")
    ex_meta  = os.path.join(existing_dir, "run_metadata.json")

    pr_final = os.path.join(primary_dir, "final_model.zip")
    pr_csv   = os.path.join(primary_dir, "episodes.csv")
    pr_meta  = os.path.join(primary_dir, "run_metadata.json")

    _copy_if_better(ex_csv, pr_csv)
    _copy_if_better(ex_meta, pr_meta)
    _copy_if_better(ex_final, pr_final)

    ex_ckpt_dir = os.path.join(existing_dir, "checkpoints")
    pr_ckpt_dir = os.path.join(primary_dir, "checkpoints")
    os.makedirs(pr_ckpt_dir, exist_ok=True)

    ex_ptr = os.path.join(ex_ckpt_dir, "latest_checkpoint.json")
    pr_ptr = os.path.join(pr_ckpt_dir, "latest_checkpoint.json")

    ex_best_ckpt, ex_best_step = pick_latest_checkpoint(ex_ckpt_dir, ckpt_prefix, ex_ptr)
    pr_best_ckpt, pr_best_step = pick_latest_checkpoint(pr_ckpt_dir, ckpt_prefix, pr_ptr)

    # Only mirror if existing has something better than primary
    if ex_best_ckpt is not None and int(ex_best_step) > int(pr_best_step):
        dst_ckpt = os.path.join(pr_ckpt_dir, os.path.basename(ex_best_ckpt))
        _copy_if_better(ex_best_ckpt, dst_ckpt)
        try:
            with open(pr_ptr, "w") as f:
                json.dump({"step": int(ex_best_step), "path": dst_ckpt, "time": time.time(), "mirrored_from": existing_dir}, f)
        except Exception:
            pass

class TimestepsCheckpointCallback(BaseCallback):
    def __init__(self, save_every_steps: int, save_path: str, name_prefix: str, pointer_path: str):
        super().__init__(verbose=0)
        self.save_every_steps = int(save_every_steps)
        self.save_path = save_path
        self.name_prefix = name_prefix
        self.pointer_path = pointer_path
        self.next_save = None

    def _on_training_start(self) -> None:
        os.makedirs(self.save_path, exist_ok=True)
        cur = int(self.model.num_timesteps)
        self.next_save = ((cur // self.save_every_steps) + 1) * self.save_every_steps
        DBG(f"[CKPT] every {self.save_every_steps:,} steps next_save={self.next_save:,}")

    def _save(self, step: int):
        stem = os.path.join(self.save_path, f"{self.name_prefix}_{step}_steps")
        self.model.save(stem)
        zip_path = stem + ".zip"
        try:
            with open(self.pointer_path, "w") as f:
                json.dump({"step": step, "path": zip_path, "time": time.time()}, f)
        except Exception:
            pass
        DBG(f"[CKPT] saved {zip_path}")

    def _on_step(self) -> bool:
        cur = int(self.model.num_timesteps)
        if self.next_save is not None and cur >= self.next_save:
            self._save(cur)
            self.next_save += self.save_every_steps
        return True

class EpisodeLoggerCallback(BaseCallback):
    """
    Paper-grade per-episode logging.
    Adds convergence speed and richer containment / fire coverage summaries.

    Convergence metric:
      first step index within episode where containment >= CONVERGENCE_CONTAINMENT_THRESH
      (None if never reached during episode)
    """
    def __init__(self, csv_path, meta_json_path, total_steps, seed, model_name, model_size, llm_model_id, llm_freq):
        super().__init__(verbose=0)
        self.csv_path = csv_path
        self.meta_json_path = meta_json_path
        self.total_steps = int(total_steps)
        self.seed = seed
        self.model_name = model_name
        self.model_size = model_size
        self.llm_model_id = llm_model_id
        self.llm_freq = int(llm_freq)

        self.start_time = None
        self.csv_file = None
        self.csv_writer = None
        self.ep_idx_global = 0
        self.global_llm_calls = 0
        self.n_envs = None
        self.buf = None

    def get_global_llm_calls(self):
        return int(self.global_llm_calls)

    def _new_buf(self):
        return {
            "r": [],
            "llm_calls": 0,
            "llm_lat": [],
            "llm_chars": 0,
            "llm_words": 0,
            "containment": [],
            "firecov": [],
            "reach_step": None,
        }

    def _init_buffers_if_needed(self):
        if self.n_envs is not None:
            return
        rewards = self.locals.get("rewards", None)
        if rewards is None:
            return
        self.n_envs = len(rewards)
        self.buf = [self._new_buf() for _ in range(self.n_envs)]
        DBG(f"[LOGGER] n_envs={self.n_envs}")

    def _open_csv(self):
        header = [
            "episode","env_idx","seed","model_name","llm_model_id","model_size","llm_freq",
            "total_steps_so_far","episode_length","timesteps_per_second","episode_return",
            "final_containment","final_fire_coverage",
            "avg_containment","max_containment",
            "avg_fire_coverage","min_fire_coverage",
            "containment_reach_step","containment_reach_threshold",
            "llm_calls","llm_chars_total","llm_words_total","avg_llm_latency_ms",
        ]
        exists = os.path.exists(self.csv_path)
        os.makedirs(os.path.dirname(self.csv_path), exist_ok=True)
        self.csv_file = open(self.csv_path, "a", newline="")
        self.csv_writer = csv.DictWriter(self.csv_file, fieldnames=header)
        if not exists:
            self.csv_writer.writeheader()
            self.csv_file.flush()

    def _on_training_start(self):
        self.start_time = time.time()
        self._open_csv()

        gpu_name = (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
        gpu_vram = float(torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0

        meta: Dict[str, Any] = {
            "experiment_name": self.model_name,
            "seed": self.seed,
            "total_steps_target": self.total_steps,
            "llm_model_id": self.llm_model_id,
            "model_size": self.model_size,
            "llm_freq": self.llm_freq,
            "env_kwargs": dict(ENV_KWARGS),
            "ppo_hyperparams": dict(PPO_HYPERPARAMS),
            "robustness_check": ROBUSTNESS_CHECK,
            "convergence_metric": "first_step_containment_ge_threshold",
            "convergence_threshold": CONVERGENCE_CONTAINMENT_THRESH,
            "ablations": dict(ABLATIONS),
            "run_notes": RUN_NOTES,
            "gpu_name": gpu_name,
            "gpu_vram_gb": gpu_vram,
            "versions": {
                "python": sys.version,
                "platform": platform.platform(),
                "torch": getattr(torch, "__version__", "unknown"),
                "stable_baselines3": getattr(__import__("stable_baselines3"), "__version__", "unknown"),
            },
            "timestamp_start": time.time(),
        }
        os.makedirs(os.path.dirname(self.meta_json_path), exist_ok=True)
        with open(self.meta_json_path, "w") as f:
            json.dump(meta, f, indent=2)

    def _flush_episode(self, env_idx: int, global_t: int, info_last: dict):
        b = self.buf[env_idx]
        self.ep_idx_global += 1

        ep_ret = float(sum(b["r"])) if b["r"] else 0.0
        ep_len = int(len(b["r"]))

        elapsed = time.time() - self.start_time
        sps = global_t / max(elapsed, 1e-6)

        cont_last = info_last.get("containment", None) if isinstance(info_last, dict) else None
        firecov_last = info_last.get("fire_coverage", None) if isinstance(info_last, dict) else None

        cont_arr = np.array(b["containment"], dtype=np.float32) if b["containment"] else None
        fire_arr = np.array(b["firecov"], dtype=np.float32) if b["firecov"] else None

        avg_cont = float(np.mean(cont_arr)) if cont_arr is not None and cont_arr.size > 0 else None
        max_cont = float(np.max(cont_arr)) if cont_arr is not None and cont_arr.size > 0 else None

        avg_fire = float(np.mean(fire_arr)) if fire_arr is not None and fire_arr.size > 0 else None
        min_fire = float(np.min(fire_arr)) if fire_arr is not None and fire_arr.size > 0 else None

        avg_lat = float(np.mean(b["llm_lat"])) if b["llm_lat"] else None

        row = {
            "episode": self.ep_idx_global,
            "env_idx": env_idx,
            "seed": self.seed,
            "model_name": self.model_name,
            "llm_model_id": self.llm_model_id,
            "model_size": self.model_size,
            "llm_freq": self.llm_freq,
            "total_steps_so_far": global_t,
            "episode_length": ep_len,
            "timesteps_per_second": sps,
            "episode_return": ep_ret,
            "final_containment": cont_last,
            "final_fire_coverage": firecov_last,
            "avg_containment": avg_cont,
            "max_containment": max_cont,
            "avg_fire_coverage": avg_fire,
            "min_fire_coverage": min_fire,
            "containment_reach_step": b["reach_step"],
            "containment_reach_threshold": CONVERGENCE_CONTAINMENT_THRESH,
            "llm_calls": int(b["llm_calls"]),
            "llm_chars_total": int(b["llm_chars"]),
            "llm_words_total": int(b["llm_words"]),
            "avg_llm_latency_ms": avg_lat,
        }
        self.csv_writer.writerow(row)
        self.csv_file.flush()

        DBG(
            f"[EP] ep={self.ep_idx_global} env={env_idx} ret={ep_ret:.2f} len={ep_len} "
            f"sps={sps:.0f} llm_calls={b['llm_calls']} reach={b['reach_step']}"
        )
        self.buf[env_idx] = self._new_buf()

    def _on_step(self):
        self._init_buffers_if_needed()
        if self.n_envs is None:
            return True

        global_t = int(self.num_timesteps)
        rewards = self.locals.get("rewards")
        infos = self.locals.get("infos")
        dones = self.locals.get("dones")

        if rewards is None or infos is None or dones is None:
            return True

        for i in range(self.n_envs):
            b = self.buf[i]
            info = infos[i] if isinstance(infos, (list, tuple)) else {}
            done = bool(dones[i])

            b["r"].append(float(rewards[i]))

            if isinstance(info, dict):
                c = info.get("containment", None)
                f = info.get("fire_coverage", None)

                if c is not None:
                    try:
                        cval = float(c)
                        b["containment"].append(cval)
                        if b["reach_step"] is None and cval >= CONVERGENCE_CONTAINMENT_THRESH:
                            b["reach_step"] = int(len(b["r"]))  # 1-based
                    except Exception:
                        pass

                if f is not None:
                    try:
                        b["firecov"].append(float(f))
                    except Exception:
                        pass

                if info.get("llm_called"):
                    b["llm_calls"] += 1
                    self.global_llm_calls += 1

                    lat = info.get("llm_latency_ms")
                    if lat is not None:
                        try:
                            b["llm_lat"].append(float(lat))
                        except Exception:
                            pass

                    ch = info.get("llm_chars")
                    if ch is not None:
                        try:
                            b["llm_chars"] += int(ch)
                        except Exception:
                            pass

                    wd = info.get("llm_words")
                    if wd is not None:
                        try:
                            b["llm_words"] += int(wd)
                        except Exception:
                            pass

            if done:
                self._flush_episode(i, global_t, info if isinstance(info, dict) else {})

        return True

    def _on_training_end(self):
        try:
            with open(self.meta_json_path, "r") as f:
                meta = json.load(f)
        except Exception:
            meta = {}

        meta.update({
            "timestamp_end": time.time(),
            "training_time_minutes": float((time.time() - self.start_time) / 60.0) if self.start_time else None,
            "episodes_total_logged": int(self.ep_idx_global),
            "llm_calls_total": int(self.global_llm_calls),
            "steps_completed": int(self.num_timesteps),
        })
        with open(self.meta_json_path, "w") as f:
            json.dump(meta, f, indent=2)

        if self.csv_file:
            self.csv_file.close()

class PinnedStatusCallback(BaseCallback):
    def __init__(self, *, exp_name, seed, llm_model_id, llm_freq, total_steps, print_every_steps, get_llm_calls_fn):
        super().__init__(verbose=0)
        self.exp_name = exp_name
        self.seed = seed
        self.llm_model_id = llm_model_id
        self.llm_freq = int(llm_freq)
        self.total_steps = int(total_steps)
        self.print_every_steps = int(print_every_steps)
        self.get_llm_calls_fn = get_llm_calls_fn
        self.t0 = None
        self.last_t = 0

    def _on_training_start(self) -> None:
        self.t0 = time.time()
        self.last_t = 0
        DBG(f"[PINNED] RUN={self.exp_name} seed={self.seed} llm={self.llm_model_id} freq={self.llm_freq} target={self.total_steps:,}")

    def _on_step(self) -> bool:
        t = int(self.model.num_timesteps)
        if t - self.last_t >= self.print_every_steps:
            self.last_t = t
            elapsed = max(time.time() - self.t0, 1e-6)
            sps = t / elapsed
            pct = 100.0 * t / max(self.total_steps, 1)
            llm_calls = int(self.get_llm_calls_fn())
            llm_per_1k = 1000.0 * llm_calls / max(t, 1)
            DBG(f"[PINNED] {self.exp_name} seed={self.seed} t={t:,}/{self.total_steps:,} ({pct:.1f}%) sps={sps:.0f} llm_calls={llm_calls:,} ({llm_per_1k:.2f}/1k)")
        return True

def make_env_for_config(cfg, seed, rank):
    def _thunk():
        env = HybridRealFireEnv(
            grid_size=ENV_KWARGS["grid_size"],
            num_drones=ENV_KWARGS["num_drones"],
            max_steps=ENV_KWARGS["max_steps"],
            llm_model=cfg["llm_model_id"],
            llm_guidance_freq=int(cfg["llm_freq"]),
            data_root=DATA_ROOT_FOR_TRAIN,
        )
        return Monitor(env)
    return _thunk

def build_fresh_ppo(env, device, seed, tb_log_dir: Optional[str], tb_log_name: str):
    kwargs = dict(
        policy="MlpPolicy",
        env=env,
        verbose=1,
        device=device,
        seed=seed,
        **PPO_HYPERPARAMS,
    )
    if TENSORBOARD_LOG and tb_log_dir is not None:
        kwargs["tensorboard_log"] = tb_log_dir
    model = PPO(**kwargs)
    model._tb_log_name = tb_log_name
    return model

def mark_complete_from_checkpoint(best_ckpt_path: str, final_model_zip: str):
    try:
        os.makedirs(os.path.dirname(final_model_zip), exist_ok=True)
        if not os.path.exists(final_model_zip):
            shutil.copy2(best_ckpt_path, final_model_zip)
            DBG(f"[COMPLETE] copied {os.path.basename(best_ckpt_path)} -> final_model.zip")
    except Exception as e:
        DBG(f"[COMPLETE][WARN] could not copy checkpoint to final_model.zip: {e}")

def run_experiment(cfg, seed):
    exp_name = cfg["name"]
    llm_model = cfg["llm_model_id"]
    model_size = cfg["model_size"]
    llm_freq = int(cfg["llm_freq"])
    ckpt_prefix = cfg["ckpt_prefix"]

    # Always create and write to PRIMARY folder (this is the fix for your Drive visibility issue)
    results_dir_primary = primary_run_dir(exp_name, seed)
    os.makedirs(results_dir_primary, exist_ok=True)

    # Find best existing run dir anywhere, then mirror best state into PRIMARY
    results_dir_existing = find_best_existing_run_dir_any_root(exp_name, seed, ckpt_prefix)
    mirror_best_state_to_primary(results_dir_existing, results_dir_primary, ckpt_prefix)

    # From this point on, ALWAYS use PRIMARY paths
    results_dir = results_dir_primary
    ckpt_dir = os.path.join(results_dir, "checkpoints")

    log_csv_path = os.path.join(results_dir, "episodes.csv")
    meta_json_path = os.path.join(results_dir, "run_metadata.json")

    final_model_zip = os.path.join(results_dir, "final_model.zip")
    pointer_path = os.path.join(ckpt_dir, "latest_checkpoint.json")

    tb_dir = os.path.join(TENSORBOARD_ROOT, exp_name, f"seed_{seed}")
    tb_name = f"{exp_name}_seed{seed}"

    print("\n" + "=" * 90, flush=True)
    print(f"[RUN] EXPERIMENT={exp_name} SEED={seed} LLM={llm_model} FREQ={llm_freq} TARGET={TOTAL_STEPS:,}", flush=True)
    print(f"[PATH] primary_results_dir ={results_dir_primary}", flush=True)
    print(f"[PATH] best_existing_dir   ={results_dir_existing}", flush=True)
    print(f"[PATH] ckpt_dir           ={ckpt_dir}", flush=True)
    print("=" * 90, flush=True)

    # 1) FASTSKIP if final_model.zip exists (in PRIMARY, after mirroring)
    if file_exists_nonempty(final_model_zip, min_bytes=MIN_FINAL_ZIP_BYTES):
        DBG("[FASTSKIP] final_model.zip exists (primary)")
        return results_dir

    # 2) FASTSKIP if checkpoint already hit TOTAL_STEPS (PRIMARY, after mirroring)
    best_ckpt, best_step = pick_latest_checkpoint(ckpt_dir, ckpt_prefix, pointer_path)
    if best_ckpt is not None and int(best_step) >= int(TOTAL_STEPS):
        DBG(f"[FASTSKIP] latest checkpoint step={best_step:,} >= target={TOTAL_STEPS:,} (primary)")
        mark_complete_from_checkpoint(best_ckpt, final_model_zip)
        return results_dir

    os.makedirs(ckpt_dir, exist_ok=True)
    if TENSORBOARD_LOG:
        os.makedirs(tb_dir, exist_ok=True)

    set_random_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    DBG(f"[DEVICE] {device}")

    env = None
    try:
        best_ckpt, best_step = pick_latest_checkpoint(ckpt_dir, ckpt_prefix, pointer_path)
        if best_ckpt:
            DBG(f"[RESUME] picked ckpt={best_ckpt} step={best_step:,} (primary)")
        else:
            DBG("[RESUME] no checkpoint found, starting fresh (primary)")

        env = DummyVecEnv([make_env_for_config(cfg, seed, rank=i) for i in range(N_ENVS)])

        if best_ckpt:
            DBG(f"[MODEL] loading ckpt={best_ckpt}")
            try:
                model = PPO.load(best_ckpt, env=env, device=device)
                model.verbose = 1
                DBG(f"[MODEL] loaded num_timesteps={int(model.num_timesteps):,}")
            except Exception as e:
                DBG(f"[MODEL][ERROR] checkpoint load failed: {e}")
                if not ALLOW_FRESH_IF_LOAD_FAIL:
                    raise RuntimeError(
                        "Checkpoint exists but failed to load. Not starting fresh (ALLOW_FRESH_IF_LOAD_FAIL=False). "
                        "Fix the load error or set ALLOW_FRESH_IF_LOAD_FAIL=True if you really want a fresh restart."
                    ) from e
                DBG("[MODEL][WARN] ALLOW_FRESH_IF_LOAD_FAIL=True so building fresh model (this will restart).")
                model = build_fresh_ppo(env, device, seed, tb_dir, tb_name)
                best_step = 0
        else:
            model = build_fresh_ppo(env, device, seed, tb_dir, tb_name)
            best_step = 0

        current_steps = max(int(getattr(model, "num_timesteps", 0)), int(best_step or 0))
        remaining_steps = max(0, TOTAL_STEPS - current_steps)
        DBG(f"[STEPS] current={current_steps:,} remaining={remaining_steps:,}")

        if remaining_steps <= 0:
            model.save(os.path.join(results_dir, "final_model"))
            DBG("[SAVE] wrote final_model.zip (already complete)")
            return results_dir

        ckpt_cb = TimestepsCheckpointCallback(CHECKPOINT_EVERY_STEPS, ckpt_dir, ckpt_prefix, pointer_path)
        episode_cb = EpisodeLoggerCallback(log_csv_path, meta_json_path, TOTAL_STEPS, seed, exp_name, model_size, llm_model, llm_freq)
        pinned_cb = PinnedStatusCallback(
            exp_name=exp_name,
            seed=seed,
            llm_model_id=llm_model,
            llm_freq=llm_freq,
            total_steps=TOTAL_STEPS,
            print_every_steps=PINNED_PRINT_EVERY_STEPS,
            get_llm_calls_fn=episode_cb.get_global_llm_calls
        )
        callbacks = CallbackList([ckpt_cb, episode_cb, pinned_cb])

        DBG("[TRAIN] starting learn()")
        learn_kwargs = dict(
            total_timesteps=remaining_steps,
            callback=callbacks,
            progress_bar=False,
            reset_num_timesteps=False,
        )
        if TENSORBOARD_LOG:
            learn_kwargs["tb_log_name"] = tb_name

        model.learn(**learn_kwargs)
        DBG("[TRAIN] learn() returned")

        model.save(os.path.join(results_dir, "final_model"))
        DBG("[SAVE] wrote final_model.zip")
        return results_dir

    except Exception as e:
        DBG(f"[FATAL] run_experiment crashed: {e}")
        traceback.print_exc()
        raise
    finally:
        if env is not None:
            try:
                env.close()
            except Exception:
                pass

DBG("=== BEGIN EXPERIMENT SWEEP ===")
for cfg in EXPERIMENTS:
    DBG(f"=== EXPERIMENT {cfg['name']} ===")
    for seed in SEEDS:
        outdir = run_experiment(cfg, seed)
        DBG(f"[POST] results_dir={outdir}")

print("Done.")
print("Primary results root:", os.path.join(RUNS_ROOT, "results"))
print("Episodes CSV pattern:", os.path.join(RUNS_ROOT, "results", "<experiment>", "seed_<seed>", "episodes.csv"))
print("Run metadata pattern:", os.path.join(RUNS_ROOT, "results", "<experiment>", "seed_<seed>", "run_metadata.json"))


Streaming output truncated to the last 5000 lines.
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[DBG 07:39:05] [EP] ep=2040 env=0 ret=27.70 len=200 sps=73 llm_calls=2 reach=1
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[DBG 07:39:07] [EP] ep=2041 env=0 ret=17.46 len=200 sps=73 llm_calls=2 reach=1
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[DBG 07:39:09] [EP] ep=2042 env=0 ret=31.45 len=200 sps=73 llm_calls=2 reach=1
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[DBG 07:39:13] [EP] ep=2043 env=0 ret=24.53 len=200 sps=73 llm_calls=2 reach=None
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[DBG 07:39:15] [EP] ep=2044 env=0 ret=28.66 len=200 sps=73 llm_calls=2 reach=1
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[QWEN] active -> 3B: Qwen/Qwen2.5-3B-Instruct
[DBG 07:39:17] [EP] ep=2045 env=0 ret=54.30 l